# Orchestrator database review

Read-only, batch-focused inspection for `orchestrator.db`. Queries open short-lived `mode=ro` connections; export is disabled unless an explicit path is configured. Column meanings live in [`docs/orchestrator_storage.md`](../../../docs/orchestrator_storage.md).

In [1]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = Path(os.getenv('ORCHESTRATOR_DB_PATH', REPO_ROOT / 'orchestrator.db'))
if not DB_PATH.is_absolute():
    DB_PATH = REPO_ROOT / DB_PATH
if not DB_PATH.is_file():
    raise FileNotFoundError(f'Database not found: {DB_PATH}')


def connect_ro() -> sqlite3.Connection:
    connection = sqlite3.connect(f'{DB_PATH.resolve().as_uri()}?mode=ro', uri=True)
    connection.row_factory = sqlite3.Row
    return connection


def query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with connect_ro() as connection:
        return pd.read_sql_query(sql, connection, params=params)


def shorten(value: object, limit: int = 180) -> object:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    text = str(value)
    return text if len(text) <= limit else text[:limit] + ' …'


def overview(frame: pd.DataFrame, blob_columns: list[str]) -> pd.DataFrame:
    trimmed = frame.copy()
    for column in blob_columns:
        if column in trimmed:
            trimmed[column] = trimmed[column].map(shorten)
    return trimmed


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 160)

## Filters

Leave `BATCH_ID` unset to inspect the most recently created batch. Limits apply to detail tables, not aggregate checks. Export remains off by default.

In [2]:
BATCH_ID = None  # e.g. 'b-...'
ROW_LIMIT = 100
DECISION_ITEM_ID = None
LOGICAL_ITEM_ID = None
EXPORT_PATH = None  # e.g. REPO_ROOT / 'output' / 'orchestrator-review.xlsx'

if ROW_LIMIT < 1:
    raise ValueError('ROW_LIMIT must be at least 1')


def active_batch_id() -> str | None:
    if BATCH_ID is not None:
        return BATCH_ID
    frame = query('SELECT batch_id FROM batches ORDER BY created_at DESC LIMIT 1')
    return None if frame.empty else str(frame.iloc[0]['batch_id'])

## Database health

Schema/version, row counts, SQLite integrity, foreign keys, and the Valid/Failure terminal invariant. A v1/v2 file is reported but never migrated by this notebook.

In [3]:
with connect_ro() as connection:
    schema_version = connection.execute('PRAGMA user_version').fetchone()[0]
    table_names = {
        row['name']
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        )
    }
    view_names = {
        row['name']
        for row in connection.execute("SELECT name FROM sqlite_master WHERE type='view'")
    }
    integrity = connection.execute('PRAGMA integrity_check').fetchone()[0]
    foreign_key_issues = connection.execute('PRAGMA foreign_key_check').fetchall()

summary = pd.DataFrame(
    [{'table': name, 'rows': int(query(f'SELECT COUNT(*) AS n FROM \"{name}\"').iloc[0]['n'])}
     for name in sorted(table_names)]
)
terminal_check = query(
    """
    SELECT COUNT(*) AS inconsistent_items
    FROM batch_items AS i
    LEFT JOIN valid_results AS v ON v.item_id=i.item_id
    LEFT JOIN failure_results AS f ON f.item_id=i.item_id
    WHERE (i.status='valid' AND (v.item_id IS NULL OR f.item_id IS NOT NULL))
       OR (i.status='failed' AND (f.item_id IS NULL OR v.item_id IS NOT NULL))
       OR (i.status IN ('pending','running') AND (v.item_id IS NOT NULL OR f.item_id IS NOT NULL))
    """
)
print(f'Database: {DB_PATH} ({DB_PATH.stat().st_size:,} bytes)')
print(f'Schema: v{schema_version}; integrity={integrity}; foreign-key issues={len(foreign_key_issues)}')
if schema_version < 3:
    print('Legacy schema: open this file once with OrchestratorDB or an orchestrator command to migrate it.')
display(summary)
display(terminal_check) 

Database: /Users/kumo/programming/competitor_product_search/orchestrator.db (258,048 bytes)
Schema: v3; integrity=ok; foreign-key issues=0


,table,rows
0,batch_items,60
1,batches,13
2,failure_results,20
3,matching_decisions,44
4,valid_results,40


,inconsistent_items
0,0


## Batch summary and lineage

In [4]:
selected_batch = active_batch_id()
if selected_batch is None:
    print('No batches in the database.')
elif 'batch_summary' not in view_names:
    display(query('SELECT * FROM batches WHERE batch_id=?', (selected_batch,)))
else:
    batch = query('SELECT * FROM batch_summary WHERE batch_id=?', (selected_batch,))
    display(overview(batch, ['job_config']))
    root_id = str(batch.iloc[0]['root_batch_id'])
    lineage = query(
        'SELECT * FROM batch_summary WHERE root_batch_id=? ORDER BY rerun_no',
        (root_id,),
    )
    display(overview(lineage, ['job_config']))

,batch_id,root_batch_id,parent_batch_id,rerun_no,operation,status,vision_enabled,source_file,job_config,created_at,finished_at,error_message,total_items,valid_count,failure_count
0,b-d02166f40e214a36982d29fd3e43af8a,b-d02166f40e214a36982d29fd3e43af8a,None,0,new_input,completed_with_failures,0,input/new_input_trial.xlsx,"{""concurrency"": 8}",2026-09-16T08:01:27.128852+00:00,2026-09-16T08:05:03.260385+00:00,None,8,5,3


,batch_id,root_batch_id,parent_batch_id,rerun_no,operation,status,vision_enabled,source_file,job_config,created_at,finished_at,error_message,total_items,valid_count,failure_count
0,b-d02166f40e214a36982d29fd3e43af8a,b-d02166f40e214a36982d29fd3e43af8a,None,0,new_input,completed_with_failures,0,input/new_input_trial.xlsx,"{""concurrency"": 8}",2026-09-16T08:01:27.128852+00:00,2026-09-16T08:05:03.260385+00:00,None,8,5,3


## Item outcomes

One row per item, combining execution state with its mutually exclusive terminal payload.

In [5]:
selected_batch = active_batch_id()
if selected_batch is None:
    outcomes = pd.DataFrame()
    print('No batches in the database.')
elif 'item_outcomes' not in view_names:
    outcomes = query(
        'SELECT * FROM batch_items WHERE batch_id=? ORDER BY row_index LIMIT ?',
        (selected_batch, ROW_LIMIT),
    )
    display(overview(outcomes, ['input_image_urls', 'stage_trace']))
else:
    outcomes = query(
        'SELECT * FROM item_outcomes WHERE batch_id=? ORDER BY row_index LIMIT ?',
        (selected_batch, ROW_LIMIT),
    )
    display(overview(outcomes, ['input_image_urls', 'stage_trace', 'product_data', 'failure_detail']))

,item_id,batch_id,logical_item_id,source_valid_result_id,row_index,input_title,country,site_name,input_gtin,input_image_urls,status,execution_path,search_title,matched_url,stage_trace,created_at,updated_at,result_id,product_data,failure_id,fail_node,failure_kind,failure_reasoning,failure_detail,terminal_at
0,53,b-d02166f40e214a36982d29fd3e43af8a,2065ab8e5e984aa4890b418ca8f47f35,None,0,Kopparberg Variety Alcohol Free 10x330ml,uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/262144/6207220124354/605064/69bb9888E65473ec8/42b10752b5083d5e.pn...",valid,new_input,Kopparberg Alcohol Free Cider Variety Pack 10 x 330ml - Tesco,https://www.tesco.com/shop/en-GB/products/315581814,"[{""at"": ""2026-09-16T08:01:50.015442+00:00"", ""stage"": ""search"", ""status"": ""success"", ""run_id"": ""a62a7aa69d184011b6dcae0b8cb94273"", ""row_index"": 0}, {""at"": ""2...",2026-09-16T08:01:27.129316+00:00,2026-09-16T08:05:03.253307+00:00,36.0,"{""url"":""https://www.tesco.com/shop/en-GB/products/315581814"",""website"":""tesco"",""scraped_at"":""2026-09-16T08:04:01.064556Z"",""source_type"":""html"",""parser_versi...",NaN,NaN,NaN,NaN,NaN,2026-09-16T08:05:03.253085+00:00
1,54,b-d02166f40e214a36982d29fd3e43af8a,2ede476e55ae4330acbc323b27f4faa8,None,1,Coca-Cola Original Taste 24 X 330ml,uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/6488064/6109817249942/266551/6903929bE44d95314/42b10752b5083d5e.p...",valid,new_input,Coca-Cola Original Taste Soft Drink Cans 24 x 330 mL,https://www.tesco.com/shop/en-GB/products/273867627,"[{""at"": ""2026-09-16T08:01:50.017033+00:00"", ""stage"": ""search"", ""status"": ""success"", ""run_id"": ""a62a7aa69d184011b6dcae0b8cb94273"", ""row_index"": 1}, {""at"": ""2...",2026-09-16T08:01:27.129764+00:00,2026-09-16T08:05:03.254684+00:00,37.0,"{""url"":""https://www.tesco.com/shop/en-GB/products/273867627"",""website"":""tesco"",""scraped_at"":""2026-09-16T08:02:27.603362Z"",""source_type"":""html"",""parser_versi...",NaN,NaN,NaN,NaN,NaN,2026-09-16T08:05:03.254642+00:00
2,55,b-d02166f40e214a36982d29fd3e43af8a,083812cf0ed24027ae18224bed2caef5,None,2,"Raid Rapid Action Wasp, Mosquito & Fly Killer Aerosol Spray 300ml",uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/2031616/61097966196984/283174/68e8b2c3E0555a310/42b10752b5083d5e....",valid,new_input,Raid Rapid Action Fly & Wasp Killer 300ml - Tesco Groceries,https://www.tesco.com/shop/en-GB/products/255284088,"[{""at"": ""2026-09-16T08:01:50.017359+00:00"", ""stage"": ""search"", ""status"": ""success"", ""run_id"": ""a62a7aa69d184011b6dcae0b8cb94273"", ""row_index"": 2}, {""at"": ""2...",2026-09-16T08:01:27.129988+00:00,2026-09-16T08:05:03.256140+00:00,38.0,"{""url"":""https://www.tesco.com/shop/en-GB/products/255284088"",""website"":""tesco"",""scraped_at"":""2026-09-16T08:02:05.451924Z"",""source_type"":""html"",""parser_versi...",NaN,NaN,NaN,NaN,NaN,2026-09-16T08:05:03.256042+00:00
3,56,b-d02166f40e214a36982d29fd3e43af8a,1a571e431f5841508b27a3f72e9d9fc8,None,3,Lynx Aerosol Bodyspray Africa 150ml,uk,tesco,None,[],valid,new_input,Lynx Africa 48h Deodorant Bodyspray for Men 150ml - Tesco,https://www.tesco.com/shop/en-GB/products/261546847,"[{""at"": ""2026-09-16T08:01:50.017668+00:00"", ""stage"": ""search"", ""status"": ""success"", ""run_id"": ""a62a7aa69d184011b6dcae0b8cb94273"", ""row_index"": 3}, {""at"": ""2...",2026-09-16T08:01:27.130204+00:00,2026-09-16T08:05:03.257516+00:00,39.0,"{""url"":""https://www.tesco.com/shop/en-GB/products/261546847"",""website"":""tesco"",""scraped_at"":""2026-09-16T08:03:11.768737Z"",""source_type"":""html"",""parser_versi...",NaN,NaN,NaN,NaN,NaN,2026-09-16T08:05:03.257450+00:00
4,57,b-d02166f40e214a36982d29fd3e43af8a,f2a75528c0134ca7971d629e09915a54,None,4,"Andrex Ultimate Quilts Toilet Tissue, 3-Ply, 16 Rolls",uk,tesco,None,"[""https://images4.joy-sourcing.com/product

## Matching decisions

In [67]:
selected_batch = active_batch_id()
if 'matching_decisions' not in table_names:
    decisions = pd.DataFrame()
    print('No matching_decisions table in this legacy database.')
elif selected_batch is None:
    decisions = pd.DataFrame()
    print('No batches in the database.')
else:
    decisions = query(
        """
        SELECT d.*, i.logical_item_id, i.row_index, i.input_title
        FROM matching_decisions AS d
        JOIN batch_items AS i ON i.item_id=d.item_id
        WHERE i.batch_id=?
        ORDER BY d.created_at desc, d.attempt_no
        LIMIT ?
        """,
        (selected_batch, ROW_LIMIT),
    )
    display(overview(decisions, ['reasoning', 'decision_process']))

,decision_id,item_id,attempt_no,execution_path,url,verdict,decision_source,gtin_status,variant_status,vision_status,reasoning,decision_process,created_at,logical_item_id,row_index,input_title
0,39,49,1,new_input,https://www.tesco.com/shop/en-GB/products/311780061,match,llm,unknown,pass,success,"Brand, Ultimate Quilts variant, family pack size of 16, and packaging imagery align; the scraped product is the same Tesco listing for Andrex Ultimate Quilt...","{""nodes"": [{""detail"": {""normalized_input_gtin"": null, ""normalized_product_gtin"": ""05029054238486""}, ""node"": ""gtin"", ""status"": ""unknown"", ""terminal"": false},...",2026-09-03T23:25:46.229617+00:00,c81f1152b9e7471fb636c565fd956ff6,4,"Andrex Ultimate Quilts Toilet Tissue, 3-Ply, 16 Rolls"
1,38,48,1,new_input,https://www.tesco.com/shop/en-GB/products/261546847,match,llm,unknown,pass,not_available,"Both refer to Lynx Africa 150ml bodyspray/deodorant with matching brand, size, and variant, and there are no conflicting details.","{""nodes"": [{""detail"": {""normalized_input_gtin"": null, ""normalized_product_gtin"": ""08717644012208""}, ""node"": ""gtin"", ""status"": ""unknown"", ""terminal"": false},...",2026-09-03T23:25:46.229027+00:00,cfbc499838ff4ea6b30c11fd4330f8b1,3,Lynx Aerosol Bodyspray Africa 150ml
2,37,47,1,new_input,https://www.tesco.com/shop/en-GB/products/255284088,match,llm,unknown,pass,success,"The Raid Rapid Action 300ml can labels, brand, and volume match, and the extra target wording in the input title does not indicate a distinct variant.","{""nodes"": [{""detail"": {""normalized_input_gtin"": null, ""normalized_product_gtin"": ""05000204795370""}, ""node"": ""gtin"", ""status"": ""unknown"", ""terminal"": false},...",2026-09-03T23:25:46.228487+00:00,1cf97743511b46a0af51510b1592bf34,2,"Raid Rapid Action Wasp, Mosquito & Fly Killer Aerosol Spray 300ml"
3,36,46,1,new_input,https://www.tesco.com/shop/en-GB/products/273867627,match,llm,unknown,pass,success,"The scraped Tesco product is Coca-Cola Original Taste 24 x 330ml cans, matching the intended SKU's brand, variant, and multipack details exactly.","{""nodes"": [{""detail"": {""normalized_input_gtin"": null, ""normalized_product_gtin"": ""05449000086877""}, ""node"": ""gtin"", ""status"": ""unknown"", ""terminal"": false},...",2026-09-03T23:25:46.227857+00:00,f3a523ad3586427191fa82c2346f56ed,1,Coca-Cola Original Taste 24 X 330ml
4,35,45,1,new_input,https://www.tesco.com/shop/en-GB/products/315581814,match,llm,unknown,pass,success,"Brand, alcohol-free cider variety pack, 10 x 330ml multipack volume all match, with no conflicting GTIN or variant-defining differences.","{""nodes"": [{""detail"": {""normalized_input_gtin"": null, ""normalized_product_gtin"": ""07333533000791""}, ""node"": ""gtin"", ""status"": ""unknown"", ""terminal"": false},...",2026-09-03T23:25:46.225204+00:00,e55e5b5544804eaf826e83111c9e05f3,0,Kopparberg Variety Alcohol Free 10x330ml


In [65]:
decisions.to_excel("matching_decisions.xlsx", index=False)

### Per-item decision replay

Set `DECISION_ITEM_ID` above, or leave it unset to replay the first item with a decision in the selected batch.

In [56]:
replay_item_id = DECISION_ITEM_ID
if replay_item_id is None and not decisions.empty:
    replay_item_id = int(decisions.iloc[0]['item_id'])
if replay_item_id is None:
    print('No Matching decision to replay.')
else:
    replay = query(
        """
        SELECT decision_id, item_id, attempt_no, execution_path, url, verdict,
               decision_source, gtin_status, variant_status, vision_status, reasoning,
               json_extract(decision_process, '$.terminated_at') AS terminated_at,
               json_array_length(decision_process, '$.nodes') AS node_count, created_at
        FROM matching_decisions WHERE item_id=? ORDER BY attempt_no
        """,
        (replay_item_id,),
    )
    display(replay)

No Matching decision to replay.


## Logical-item history

In [55]:
logical_id = LOGICAL_ITEM_ID
if logical_id is None and not outcomes.empty:
    logical_id = str(outcomes.iloc[0]['logical_item_id'])
if logical_id is None:
    print('No logical item to inspect.')
elif 'item_outcomes' not in view_names:
    display(query('SELECT * FROM batch_items WHERE logical_item_id=? ORDER BY item_id', (logical_id,)))
else:
    history = query(
        """
        SELECT o.*, b.operation, b.rerun_no
        FROM item_outcomes AS o JOIN batches AS b ON b.batch_id=o.batch_id
        WHERE o.logical_item_id=? ORDER BY o.item_id
        """,
        (logical_id,),
    )
    display(overview(history, ['input_image_urls', 'stage_trace', 'product_data', 'failure_detail']))

,item_id,batch_id,logical_item_id,source_valid_result_id,row_index,input_title,country,site_name,input_gtin,input_image_urls,status,execution_path,search_title,matched_url,stage_trace,created_at,updated_at,result_id,product_data,failure_id,fail_node,failure_kind,failure_reasoning,failure_detail,terminal_at,operation,rerun_no
0,41,b-2456586aad9d4ec5b74fed86bb140f11,6ae87cca13c14b62b033ab9adfc4ebd2,None,0,Kopparberg Variety Alcohol Free 10x330ml,uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/262144/6207220124354/605064/69bb9888E65473ec8/42b10752b5083d5e.pn...",failed,new_input,Kopparberg Alcohol Free Cider Variety Pack 10 x 330ml - Tesco,https://www.tesco.com/shop/en-GB/products/315581814,"[{""at"": ""2026-09-03T20:55:21.328799+00:00"", ""stage"": ""search"", ""status"": ""success"", ""run_id"": ""3f5aa08ed5ed4381a6bc7d8250bad63b""}, {""at"": ""2026-09-03T21:01:...",2026-09-03T20:53:59.292630+00:00,2026-09-03T21:02:40.912894+00:00,None,None,14,match,no_match,"The visual evidence shows different flavour assortments: the input pack contains Summer Punch but no Crisp Apple, while the scraped listing contains Crisp A...",{},2026-09-03T21:02:40.912249+00:00,new_input,0


## Latest Valid snapshots

In [54]:
if 'latest_valid_results' not in view_names:
    print('latest_valid_results is available after migration to schema v3.')
else:
    latest = query(
        'SELECT * FROM latest_valid_results ORDER BY logical_item_id LIMIT ?',
        (ROW_LIMIT,),
    )
    display(overview(latest, ['input_image_urls', 'product_data']))

,result_id,item_id,batch_id,root_batch_id,operation,rerun_no,logical_item_id,input_title,country,site_name,input_gtin,input_image_urls,search_title,url,execution_path,product_data,created_at
0,5,14,b-622f110a40474894911f4dfe850d78d2,b-622f110a40474894911f4dfe850d78d2,new_input,0,0c39174d669048f7a19892bec34d3ec1,Coca-Cola Original Taste 24 X 330ml,uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/6488064/6109817249942/266551/6903929bE44d95314/42b10752b5083d5e.p...",Coca-Cola Original Taste Soft Drink Cans 24 x 330 mL - Tesco ...,https://www.tesco.com/shop/en-GB/products/266300013,new_input,"{""url"":""https://www.tesco.com/shop/en-GB/products/266300013"",""website"":""tesco"",""scraped_at"":""2026-09-02T22:35:10.890719Z"",""source_type"":""html"",""parser_versi...",2026-09-02T22:39:03.723503+00:00
1,23,36,b-47b68ecccdb04fb998088279cf37716a,b-47b68ecccdb04fb998088279cf37716a,new_input,0,0ea576a736904b37a713a81afeaaf6c5,Lynx Aerosol Bodyspray Africa 150ml,uk,tesco,None,[],Lynx Africa 48h Deodorant Bodyspray for Men 150ml - Tesco,https://www.tesco.com/shop/en-GB/products/261546847,new_input,"{""url"":""https://www.tesco.com/shop/en-GB/products/261546847"",""website"":""tesco"",""scraped_at"":""2026-09-03T20:19:13.327528Z"",""source_type"":""html"",""parser_versi...",2026-09-03T20:20:09.704344+00:00
2,14,25,b-f2d2ab62a9fa490cbcc6a9266a20e512,b-f2d2ab62a9fa490cbcc6a9266a20e512,new_input,0,1cd55fe6ef0f48c4ba296057844a7558,Kopparberg Variety Alcohol Free 10x330ml,uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/262144/6207220124354/605064/69bb9888E65473ec8/42b10752b5083d5e.pn...",Kopparberg Alcohol Free Cider Variety Pack 10 x 330ml - Tesco,https://www.tesco.com/shop/en-GB/products/315581814,new_input,"{""url"":""https://www.tesco.com/shop/en-GB/products/315581814"",""website"":""tesco"",""scraped_at"":""2026-09-03T01:13:28.157161Z"",""source_type"":""html"",""parser_versi...",2026-09-03T01:14:20.176127+00:00
3,2,8,b-3c0a592eb94e4187b415e9286edea41f,b-3c0a592eb94e4187b415e9286edea41f,new_input,0,1cf410cdaecb45f38df0ec8092d1025c,Lynx Aerosol Bodyspray Africa 150ml,uk,tesco,None,[],Lynx Africa 48h Deodorant Bodyspray for Men 150ml - Tesco,https://www.tesco.com/shop/en-GB/products/261546847,new_input,"{""url"":""https://www.tesco.com/shop/en-GB/products/261546847"",""website"":""tesco"",""scraped_at"":""2026-09-02T22:18:39.972954Z"",""source_type"":""html"",""parser_versi...",2026-09-02T22:18:41.943223+00:00
4,26,39,b-4aa3ec3d68384c3da01d6e807e3e5791,b-4aa3ec3d68384c3da01d6e807e3e5791,new_input,0,46680991c59a46dbbe1f45135fc3bfa3,"Raid Rapid Action Wasp, Mosquito & Fly Killer Aerosol Spray 300ml",uk,tesco,None,"[""https://images4.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-product/t1/4294967296/2031616/61097966196984/283174/68e8b2c3E0555a310/42b10752b5083d5e....",Raid Rapid Action Fly & Wasp Killer 300ml - Tesco Groceries,https://www.tesco.com/shop/en-GB/products/255284088,new_input,"{""url"":""https://www.tesco.com/shop/en-GB/products/255284088"",""website"":""tesco"",""scraped_at"":""2026-09-03T20:43:33.656083Z"",""source_type"":""html"",""parser_versi...",2026-09-03T20:43:43.927652+00:00
5,7,16,b-622f110a40474894911f4dfe850d78d2,b-622f110a40474894911f4dfe850d78d2,new_input,0,49d0d962c9a846eda03d1712d1e2f10a,Lynx Aerosol Bodyspray Africa 150ml,uk,tesco,None,[],Lynx Africa 48h Deodorant Bodyspray for Men 150ml - Tesco,https://www.tesco.com/shop/en-GB/products/261546847,new_input,"{""url"":""https://www.tesco.com/shop/en-GB/products/261546847"",""website"":""tesco"",""scraped_at"":""2026-09-02T22:35:29.983839Z"",""source_type"":""html"",""parser_versi...",2026-09-02T22:39:03.724709+00:00
6,19,31,b-c143376512b44684b14d29aac5e2d68b,b-c143376512b44684b14d29aac5e2d68b,new_input,0,4a55d5456e114f9e8e3b03e2ed2c747e,"Raid Rapid Action Wasp, Mosquito & Fly Killer Aerosol Spray 300ml",uk,tesco,None,"[""https://images4.joy-sourci

## Failure breakdown

In [52]:
selected_batch = active_batch_id()
if selected_batch is None:
    print('No batches in the database.')
else:
    breakdown = query(
        """
        SELECT b.operation, f.fail_node, f.failure_kind, COUNT(*) AS failures
        FROM failure_results AS f
        JOIN batch_items AS i ON i.item_id=f.item_id
        JOIN batches AS b ON b.batch_id=i.batch_id
        WHERE i.batch_id=?
        GROUP BY b.operation, f.fail_node, f.failure_kind
        ORDER BY failures DESC, f.fail_node, f.failure_kind
        """,
        (selected_batch,),
    )
    display(breakdown)

,operation,fail_node,failure_kind,failures
0,new_input,match,no_match,1


## Full JSON drill-down

Overview tables truncate JSON. `show_blob` prints one complete field without keeping a database handle open.

In [53]:
REVIEW_TABLES = {
    'batches': 'batch_id',
    'batch_items': 'item_id',
    'valid_results': 'result_id',
    'failure_results': 'failure_id',
    'matching_decisions': 'decision_id',
}


def show_blob(table: str, row_id: object, column: str) -> None:
    if table not in REVIEW_TABLES or table not in table_names:
        raise ValueError(f'Unknown review table: {table}')
    with connect_ro() as connection:
        valid_columns = {row['name'] for row in connection.execute(f'PRAGMA table_info({table})')}
        if column not in valid_columns:
            raise ValueError(f'Unknown column for {table}: {column}')
        key = REVIEW_TABLES[table]
        row = connection.execute(
            f'SELECT \"{column}\" FROM {table} WHERE {key}=?', (row_id,)
        ).fetchone()
    if row is None:
        raise LookupError(f'No {table} row with {key}={row_id!r}')
    value = row[0]
    if value is None:
        print('(NULL)')
        return
    try:
        print(json.dumps(json.loads(value), indent=2, ensure_ascii=False))
    except (TypeError, json.JSONDecodeError):
        print(value)


# Examples:
# show_blob('batch_items', 1, 'stage_trace')
# show_blob('matching_decisions', 1, 'decision_process')
# show_blob('valid_results', 1, 'product_data')
# show_blob('failure_results', 1, 'detail')

## Optional export

No file is written when `EXPORT_PATH` is `None`. Set an explicit `.xlsx` path in the Filters cell to export the selected batch.

In [ ]:
if EXPORT_PATH is None:
    print('Export disabled.')
else:
    export_path = Path(EXPORT_PATH).expanduser().resolve()
    if export_path.suffix.lower() != '.xlsx':
        raise ValueError('EXPORT_PATH must end in .xlsx')
    export_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(export_path) as writer:
        outcomes.to_excel(writer, sheet_name='item_outcomes', index=False)
        decisions.to_excel(writer, sheet_name='matching_decisions', index=False)
        breakdown.to_excel(writer, sheet_name='failure_breakdown', index=False)
    print(f'Exported: {export_path}')